# Step 2: Load All Files Automatically
This cell scans the raw data folder and loads every supported file into a dictionary of dataframes.

In [7]:
from pathlib import Path
import pandas as pd
import re

data_dir = (Path.cwd().parent / "data" / "raw").resolve()
supported_ext = {".csv", ".xlsx", ".xls", ".parquet"}

year_file_pattern = re.compile(r"amazon_india_\d{4}\.(csv|xlsx|xls|parquet)$", re.IGNORECASE)

file_paths = sorted(
    p
    for p in data_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in supported_ext
    and year_file_pattern.search(p.name)
 )

def load_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path}")

if not file_paths:
    print(f"No supported year files found in: {data_dir}")
    dataframes = {}
else:
    dataframes = {p.name: load_file(p) for p in file_paths}
    print(f"Loaded {len(dataframes)} year files from {data_dir}")
    file_summary = {name: df.shape for name, df in dataframes.items()}
    for name, shape in file_summary.items():
        print(f"{name}: {shape}")

    file_summary

Loaded 11 year files from C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\raw
amazon_india_2015.csv: (33165, 34)
amazon_india_2016.csv: (55275, 34)
amazon_india_2017.csv: (77385, 34)
amazon_india_2018.csv: (99495, 34)
amazon_india_2019.csv: (121605, 34)
amazon_india_2020.csv: (143715, 34)
amazon_india_2021.csv: (138187, 34)
amazon_india_2022.csv: (132660, 34)
amazon_india_2023.csv: (127132, 34)
amazon_india_2024.csv: (121605, 34)
amazon_india_2025.csv: (77385, 34)


In [8]:
# Merge all year files and report shape
if dataframes:
    merged_df = pd.concat(dataframes.values(), ignore_index=True)
    print(f"Merged rows/cols: {merged_df.shape}")
else:
    merged_df = pd.DataFrame()
    print("No files to merge.")

Merged rows/cols: (1127609, 34)


In [20]:
# Keep an in-memory copy of raw merged data
if not merged_df.empty:
    raw_df = merged_df.copy()
    print(f"Raw copy created: {raw_df.shape}")
else:
    raw_df = pd.DataFrame()
    print("Merged dataset is empty; raw copy not created.")

Raw copy created: (1127609, 34)


In [ ]:
# Save merged dataset to raw folder
cleaned_dir = (Path.cwd().parent / "data" / "raw").resolve()
cleaned_dir.mkdir(parents=True, exist_ok=True)
merged_path = cleaned_dir / "amazon_india_all_years.csv"

if not cleaned_df.empty:
    cleaned_df.to_csv(merged_path, index=False)
    print(f"Saved cleaned merged dataset to: {merged_path}")
else:
    print("Cleaned dataset is empty; nothing saved.")

Saved merged dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\raw\amazon_india_all_years.csv


In [9]:
# Column names per year and extra columns across files
if dataframes:
    print("Columns per file:")
    for name, df in dataframes.items():
        print(f"{name} ({df.shape[1]} cols)")
        print(df.columns.tolist())
        print("-")

    all_cols = set().union(*(df.columns for df in dataframes.values()))
    common_cols = set.intersection(*(set(df.columns) for df in dataframes.values()))
    extra_cols = sorted(all_cols - common_cols)
    print("Extra columns (not in all files):")
    print(extra_cols)
else:
    print("No files loaded.")

Columns per file:
amazon_india_2015.csv (34 cols)
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']
-
amazon_india_2016.csv (34 cols)
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending

In [16]:
# Product catalog: load, shape, and columns
catalog_path = data_dir / "amazon_india_products_catalog.csv"

if catalog_path.exists():
    catalog_df = pd.read_csv(catalog_path)
    print(f"Catalog shape: {catalog_df.shape}")
    print("Catalog columns:")
    for col in catalog_df.columns:
        print(f"- {col}")
else:
    catalog_df = pd.DataFrame()
    print(f"Catalog file not found: {catalog_path}")

Catalog shape: (2004, 11)
Catalog columns:
- product_id
- product_name
- category
- subcategory
- brand
- base_price_2015
- weight_kg
- rating
- is_prime_eligible
- launch_year
- model


## Data Summary
1. Each year file name, rows, columns, shape:
   - amazon_india_2015.csv: (33165, 34)
   - amazon_india_2016.csv: (55275, 34)
   - amazon_india_2017.csv: (77385, 34)
   - amazon_india_2018.csv: (99495, 34)
   - amazon_india_2019.csv: (121605, 34)
   - amazon_india_2020.csv: (143715, 34)
   - amazon_india_2021.csv: (138187, 34)
   - amazon_india_2022.csv: (132660, 34)
   - amazon_india_2023.csv: (127132, 34)
   - amazon_india_2024.csv: (121605, 34)
   - amazon_india_2025.csv: (77385, 34)
2. Total of all year data (merged):
   - Total records = 1127609
   - Total columns = 34
   - Shape = (1127609, 34)
3. Column names:
   - transaction_id
   - order_date
   - customer_id
   - product_id
   - product_name
   - category
   - subcategory
   - brand
   - original_price_inr
   - discount_percent
   - discounted_price_inr
   - quantity
   - subtotal_inr
   - delivery_charges
   - final_amount_inr
   - customer_city
   - customer_state
   - customer_tier
   - customer_spending_tier
   - customer_age_group
   - payment_method
   - delivery_days
   - delivery_type
   - is_prime_member
   - is_festival_sale
   - festival_name
   - customer_rating
   - return_status
   - order_month
   - order_year
   - order_quarter
   - product_weight_kg
   - is_prime_eligible
   - product_rating

In [15]:
# Auto-fill markdown summary values
if dataframes:
    year_lines = [
        f"   - {name}: {df.shape}"
        for name, df in dataframes.items()
    ]
    merged_df = pd.concat(dataframes.values(), ignore_index=True)
    merged_shape = merged_df.shape
    col_lines = [f"   - {col}" for col in merged_df.columns.tolist()]
else:
    year_lines = ["   - ___"]
    merged_shape = (0, 0)
    col_lines = ["   - ___"]

summary_md = "\n".join([
    "## Data Summary",
    "1. Each year file name, rows, columns, shape:",
    *year_lines,
    "2. Total of all year data (merged):",
    f"   - Total records = {merged_shape[0]}",
    f"   - Total columns = {merged_shape[1]}",
    f"   - Shape = {merged_shape}",
    "3. Column names:",
    *col_lines,
])
print(summary_md)

## Data Summary
1. Each year file name, rows, columns, shape:
   - amazon_india_2015.csv: (33165, 34)
   - amazon_india_2016.csv: (55275, 34)
   - amazon_india_2017.csv: (77385, 34)
   - amazon_india_2018.csv: (99495, 34)
   - amazon_india_2019.csv: (121605, 34)
   - amazon_india_2020.csv: (143715, 34)
   - amazon_india_2021.csv: (138187, 34)
   - amazon_india_2022.csv: (132660, 34)
   - amazon_india_2023.csv: (127132, 34)
   - amazon_india_2024.csv: (121605, 34)
   - amazon_india_2025.csv: (77385, 34)
2. Total of all year data (merged):
   - Total records = 1127609
   - Total columns = 34
   - Shape = (1127609, 34)
3. Column names:
   - transaction_id
   - order_date
   - customer_id
   - product_id
   - product_name
   - category
   - subcategory
   - brand
   - original_price_inr
   - discount_percent
   - discounted_price_inr
   - quantity
   - subtotal_inr
   - delivery_charges
   - final_amount_inr
   - customer_city
   - customer_state
   - customer_tier
   - customer_spending_t